# OD Model Benchmark

Comparison pipeline: **Our GPS-based model** (pre-trained in `gps_od.ipynb`) vs
**TransFlower orig** (paper baseline) and classical/graph/diffusion baselines.

**Our model:** GPS-GNN encoder + decoder — loaded from pre-trained weights.
**Baseline (paper):** TransFlower orig (MLP encoder + TransFlower decoder + RLE) — trained inline.
**Baselines:** RF, SVR, GBRT, DGM, GM_E, GM_P, GMEL — trained inline.


In [1]:
from pathlib import Path

import pandas as pd

from benchmarking.config import (
    BASELINE_MODELS,
    DATA_PATH,
    GPS_BENCHMARK_IDS,
    GPS_MC_BENCHMARK_IDS,
    MULTI_CITY_IDS,
    PROJECT_ROOT,
    SINGLE_CITY_ID,
    TRANSFLOWER_ORIG_CONFIG,
    WEIGHTS_DIR,
    cleanup_gpu,
    device,
    set_global_seed,
    trained_gps_run_ids,
    trained_lgbm_run_ids,
)
from benchmarking.data_utils import get_all_areas
from benchmarking.gps_loader import GPSBenchmarkLoader
from benchmarking.pipeline import run_multi_city_benchmark, run_single_city_benchmark
from benchmarking.reporting import (
    build_combined_summary,
    plot_comparison,
    results_to_dataframe,
    save_results_table,
)

set_global_seed()

trained_sc = trained_gps_run_ids(GPS_BENCHMARK_IDS)
trained_mc = trained_gps_run_ids(GPS_MC_BENCHMARK_IDS)
LGBM_BENCHMARK_IDS = trained_lgbm_run_ids(GPS_BENCHMARK_IDS)
gps_loader = GPSBenchmarkLoader(single_city_id=SINGLE_CITY_ID, multi_city_ids=MULTI_CITY_IDS, data_path=DATA_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(f"Data path: {DATA_PATH}")


Project root: /home/rk/Документы/Python Projects/GSP_OD_Prediction
Device: cuda:1
Data path: /home/rk/Документы/Python Projects/GSP_OD_Prediction/data


## Configurable Model List
Comment/uncomment to select which models to run.

In [2]:
print(f"GPS (SC) run_ids: {len(GPS_BENCHMARK_IDS)}  ({len(trained_sc)} trained)")
print(f"GPS (MC) run_ids: {len(GPS_MC_BENCHMARK_IDS)}  ({len(trained_mc)} trained)")
print(f"LGBM run_ids:     {len(LGBM_BENCHMARK_IDS)}")
print(f"Baseline models:  {len(BASELINE_MODELS)}")
print(f"TransFlower orig config: {TRANSFLOWER_ORIG_CONFIG.describe()}")


GPS (SC) run_ids: 12  (9 trained)
GPS (MC) run_ids: 7  (6 trained)
Baseline models:  7


## Shared Data Utilities

In [3]:
print(f"Total areas in dataset: {len(get_all_areas(DATA_PATH))}")


Total areas in dataset: 2265


## Model Runners

One runner function per model family. Each returns `list[dict]` of per-area metrics.

In [4]:
from benchmarking.runners import load_model_main, run_diffusion_model, run_flat_model, run_graph_model

REUSED_MODEL_TRAINS = ["DGM", "GM_E", "GM_P", "GMEL"]
print("Benchmark helpers loaded from benchmarking/*.py")
print(f"Reusable model entrypoints: {', '.join(REUSED_MODEL_TRAINS)}")


Flat model runner defined (RF/SVR/GBRT chunked; DGM/GM_E/GM_P via model main.py).


In [5]:
print("Graph-family runner is provided by benchmarking.runners.run_graph_model")


Graph model runner defined (GMEL via model main.py).


In [6]:
print("Diffusion-family runner is provided by benchmarking.runners.run_diffusion_model")


Diffusion model runner defined.


In [7]:
print("GPS and GPS+LGBM loading is provided by benchmarking.gps_loader.GPSBenchmarkLoader")


GPS results loader defined.


## Single-City Benchmark

All models predict on city `48201`. OD pairs split 80/10/10 for training.

In [8]:
print("=" * 70)
print("  SINGLE-CITY BENCHMARK")
print(f"  City: {SINGLE_CITY_ID}")
print("=" * 70)

single_city_results, single_city_model_type = run_single_city_benchmark(
    gps_run_ids=GPS_BENCHMARK_IDS,
    lgbm_run_ids=LGBM_BENCHMARK_IDS,
    single_city_id=SINGLE_CITY_ID,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
)

print(f"\nCompleted: {len(single_city_results)} models")


  SINGLE-CITY BENCHMARK
  City: 48201

[Our Model — GPS variants]
  Loading SC_TF_CE (config from JSON) ...
  SC_TF_CE: CPC=0.7051  MAE=1.5131
  Loading SC_TF_H (config from JSON) ...
  SC_TF_H: CPC=0.6754  MAE=1.6827
  Loading SC_TF_CE_lape (config from JSON) ...
  SC_TF_CE_lape: CPC=0.3282  MAE=3.4478
  Loading SC_TF_CE_gn (config from JSON) ...
  SC_TF_CE_gn: CPC=0.6925  MAE=1.5779
  Loading SC_TF_focal (config from JSON) ...
  SC_TF_focal: CPC=0.7083  MAE=1.4969
  Loading SC_TF_CE_samp (config from JSON) ...
  SC_TF_CE_samp: CPC=0.7011  MAE=1.5340
  Loading SC_TF_CE_nz (config from JSON) ...
  SC_TF_CE_nz: CPC=0.6720  MAE=1.6834
  [SKIP] SC_TF_CE_rle: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_CE_rle.pt
         -> Run gps_od.ipynb first to train this configuration.
  [SKIP] SC_TF_CE_lape_rle: weights not found at /home/rk/Документы/Python Projects/GSP_OD_Prediction/results/weights/SC_TF_CE_lape_rle.pt
         -> Run gps_od.ipyn

RF chunks: 100%|██████████| 1/1 [05:50<00:00, 350.73s/it]


    Chunk 1/1: 617796 samples, 20 trees
  RF trained: 20 trees total
  CPC=0.7050  RMSE=3.89  MAE=1.51  (352.0s)

  Running: SVR


SVR scaler: 100%|██████████| 1/1 [00:01<00:00,  1.92s/it]


  Scaler fitted on 617796 total samples


...

  CPC=0.3090  RMSE=10.43  MAE=2.40  (15.2s)

  Running: GBRT


GBRT load: 100%|██████████| 1/1 [00:00<00:00,  2.04it/s]


  GBRT training on 617796 samples (650 MB)
  CPC=0.6773  RMSE=4.00  MAE=1.66  (25.1s)

  Running: DGM


DGM:   1%|          | 100/10000 [16:11<26:43:12,  9.72s/ep, loss=0.0004499, pat=1, val=0.0004499]


  CPC=0.0000  RMSE=10.84  MAE=2.57  (975.2s)

  Running: GM_E
  ERROR GM_E: cannot import name 'GRAVITY' from 'model' (/home/rk/Документы/Python Projects/GSP_OD_Prediction/models/DGM/model.py)

  Running: GM_P
  ERROR GM_P: cannot import name 'GRAVITY' from 'model' (/home/rk/Документы/Python Projects/GSP_OD_Prediction/models/DGM/model.py)

  Running: GMEL
  ERROR GMEL: cannot import name 'GMEL' from 'model' (/home/rk/Документы/Python Projects/GSP_OD_Prediction/models/DGM/model.py)

Completed: 13 models


In [9]:
if single_city_results:
    df_single = results_to_dataframe(single_city_results, single_city_model_type)

    print("\n" + "=" * 100)
    print("  SINGLE-CITY RESULTS (sorted by CPC)")
    print("=" * 100)
    print(df_single.to_string())

    single_city_path = save_results_table(df_single, "single_city_benchmark.csv")
    print(f"\nSaved to {single_city_path}")
else:
    print("No results to display.")



  SINGLE-CITY RESULTS (sorted by CPC)
                    CPC  CPC_nonzero       RMSE       MAE      MAPE     SMAPE  accuracy  matrix_COS_similarity  JSD_inflow  JSD_outflow  JSD_ODflow
SC_BL_CE       0.733683     0.778604   3.940314  1.366696  0.482367  1.361163  0.774414               0.801458    0.006181     0.000000    0.002570
SC_TF_focal    0.708309     0.755600   4.532167  1.496909  0.513895  1.368850  0.766250               0.781430    0.024900     0.000000    0.002843
SC_TF_CE       0.705147     0.754632   4.588737  1.513137  0.530660  1.365166  0.764400               0.781428    0.025488     0.000000    0.002178
RF             0.704959     0.778681   3.887369  1.513537  0.656831  1.339560  0.710709               0.844016    0.203013     0.008038    0.004809
SC_TF_CE_samp  0.701081     0.762323   4.508370  1.534006  0.582828  1.317830  0.767227               0.807841    0.054264     0.000000    0.004712
SC_TF_CE_gn    0.692529     0.741705   5.156372  1.577892  0.536941  1.35

## Multi-City (10 Cities) Benchmark

10 cities, area-level split: 8 train, 1 val, 1 test.

In [ ]:
print("=" * 70)
print("  MULTI-CITY BENCHMARK")
print(f"  Cities: {MULTI_CITY_IDS}")
print("=" * 70)

multi_city_results, multi_city_model_type, multi_city_split = run_multi_city_benchmark(
    gps_run_ids=GPS_MC_BENCHMARK_IDS,
    city_ids=MULTI_CITY_IDS,
    data_path=DATA_PATH,
    baseline_models=BASELINE_MODELS,
    gps_loader=gps_loader,
)

print(f"Train: {multi_city_split['train']}")
print(f"Valid: {multi_city_split['valid']}")
print(f"Test:  {multi_city_split['test']}")
print(f"\nCompleted: {len(multi_city_results)} models")


  MULTI-CITY BENCHMARK
  Cities: ['17031', '48201', '04013', '06073', '06059', '36047', '12086', '48113', '06065', '36081']
Train: ['06065', '48201', '36047', '17031', '48113', '04013', '36081', '06059']
Valid: ['06073']
Test:  ['12086']

[Our Model — GPS variants]
  17031: N=1318
  48201: N=786
  04013: N=916
  06073: N=627
  06059: N=582
  36047: N=761
  12086: N=518
  48113: N=529
  06065: N=453
  36081: N=669
  06065: N=453
  48201: N=786
  36047: N=761
  17031: N=1318
  48113: N=529
  04013: N=916
  36081: N=669
  06059: N=582
  06073: N=627
  12086: N=518
  Loading MC_TF_CE (config from JSON) ...
  MC_TF_CE: CPC=0.3379  MAE=2.6325
  Loading MC_TF_CE (config from JSON) ...
  MC_TF_CE: CPC=0.6471  MAE=1.8108
  Loading MC_TF_CE (config from JSON) ...
  MC_TF_CE: CPC=0.3734  MAE=0.8136
  Loading MC_TF_CE (config from JSON) ...
  MC_TF_CE: CPC=0.4149  MAE=1.2300
  Loading MC_TF_CE (config from JSON) ...
  MC_TF_CE: CPC=0.5647  MAE=2.4221
  Loading MC_TF_CE (config from JSON) ...
  MC_

In [ ]:
if multi_city_results:
    df_multi = results_to_dataframe(multi_city_results, multi_city_model_type)

    print("\n" + "=" * 100)
    print("  MULTI-CITY RESULTS (sorted by CPC)")
    print("=" * 100)
    print(df_multi.to_string())

    multi_city_path = save_results_table(df_multi, "multi_city_benchmark.csv")
    print(f"\nSaved to {multi_city_path}")
else:
    print("No results to display.")


## Visualization

In [ ]:
plot_comparison(single_city_results, f"Single-City Benchmark ({SINGLE_CITY_ID})", ["CPC", "MAE", "RMSE"])
plot_comparison(multi_city_results, "Multi-City Benchmark (10 cities)", ["CPC", "MAE", "RMSE"])


In [ ]:
print("\n" + "=" * 120)
print("  COMBINED SUMMARY")
print("=" * 120)

df_summary = build_combined_summary(single_city_results, multi_city_results)
print(df_summary.to_string())

summary_path = save_results_table(df_summary, "benchmark_combined.csv")
print(f"\nSaved to {summary_path}")
